# Nehalennia data layer

Harvests each source into a dataframe, then writes the columns RMO and RMA share to
`presentation_layer/data/objecten.json`.

In [ ]:
# %pip install pandas openpyxl rdflib

import pandas as pd

import enrich
import links
import source_collectie_zeeland
import source_rma
import source_rmo
from const import CACHE_DIR, DATA_DIR, make_cache_dir

make_cache_dir(CACHE_DIR)

OBJECTEN_PATH = DATA_DIR / "objecten.json"
COLUMNS = [
    "pid",
    "bron",
    "objectnummer",
    "titel",
    "objectnaam",
    "beschrijving",
    "vindplaats",
    "datering_datum_start",
    "datering_datum_eind",
    "gevonden_datum_van",
    "gevonden_datum_tot",
    "image_url",
    "inscripties_text",
    "inscripties",
    "edcs_vertaling_concept",
    "period_ids",
    "linked_objecten",
]


def write_json(path, frames, columns=COLUMNS):
    """COLUMNS in that order, whatever a frame is missing comes out null"""
    objecten = pd.concat([frame.reindex(columns=columns) for frame in frames],
                         ignore_index=True)

    # escape float nan
    for column in columns:
        owners = [frame[column] for frame in frames if column in frame.columns]
        if owners and all(pd.api.types.is_integer_dtype(owner) for owner in owners):
            objecten[column] = objecten[column].astype("Int64")

    # most linked first, so the richest objects lead; stable keeps runs diffable
    objecten = objecten.sort_values(
        ["linked_objecten", "bron"],
        ascending=[False, True],
        key=lambda column: column.map(len) if column.name == "linked_objecten" else column,
        kind="stable",
    ).reset_index(drop=True)

    return write_records(path, objecten)


def write_records(path, frame):
    """One JSON array, one object per row"""
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_json(path, orient="records", indent=2, force_ascii=False)
    return frame

## RMO

In [ ]:
df_rmo, df_rmo_images = source_rmo.fetch(cache_dir=CACHE_DIR)
df_rmo.head()

## RMA

In [ ]:
df_rma = source_rma.fetch(cache_dir=CACHE_DIR)
df_rma[["objectnummer", "titel", "image_url", "edcs_vertaling_concept"]].head()

## Collectie Zeeland

Streams the dump on every run (no cache)

In [ ]:
df_zeeland = source_collectie_zeeland.fetch(ttl_path=CACHE_DIR / "zeeland_nehalennia.ttl")
df_zeeland[["objectnummer", "titel", "objecttype", "materiaal", "vervaardiger"]]

## Export

In [ ]:
df_rmo = enrich.periods(df_rmo)
df_rma = enrich.periods(df_rma)

df_rmo, df_rma = links.attach([df_rmo, df_rma])

objecten = write_json(OBJECTEN_PATH, [df_rmo, df_rma])
objecten.head()